# 04 - Business Questions

## Objetivo

Responder às perguntas de negócio do case utilizando exclusivamente as tabelas da camada Gold.

In [0]:
# Leitura das tabelas da camada Gold

fact_sales = spark.table("workspace.gold.fact_sales")
dim_brand = spark.table("workspace.gold.dim_brand")
dim_channel = spark.table("workspace.gold.dim_channel")
dim_region = spark.table("workspace.gold.dim_region")
dim_date = spark.table("workspace.gold.dim_date")

print("Fact Sales:", fact_sales.count())

Fact Sales: 16151


In [0]:
# Top 3 Trade Groups por região em volume de vendas

from pyspark.sql.functions import *
from pyspark.sql.window import Window

top_trade_groups_df = (
    fact_sales.alias("f")
    .join(dim_region.alias("r"), "region_key")
    .join(dim_channel.alias("c"), "channel_key")
    .groupBy(
        "r.Btlr_Org_LVL_C_Desc",
        "c.TRADE_GROUP_DESC"
    )
    .agg(
        sum("f.sales_volume").alias("total_sales_volume")
    )
)

window_region = Window.partitionBy(
    "Btlr_Org_LVL_C_Desc"
).orderBy(
    col("total_sales_volume").desc()
)

top_trade_groups_df = (
    top_trade_groups_df
    .withColumn("ranking", row_number().over(window_region))
    .filter(col("ranking") <= 3)
    .orderBy("Btlr_Org_LVL_C_Desc", "ranking")
)

display(top_trade_groups_df)

Btlr_Org_LVL_C_Desc,TRADE_GROUP_DESC,total_sales_volume,ranking
CANADA,GROCERY,168640.07000000007,1
CANADA,SERVICES,83748.06999999998,2
CANADA,ACADEMIC,53117.56000000003,3
GREAT LAKES,GROCERY,380297.1899999994,1
GREAT LAKES,SERVICES,157798.63999999984,2
GREAT LAKES,ACADEMIC,152043.01000000004,3
MIDWEST,GROCERY,326351.38999999943,1
MIDWEST,SERVICES,129258.59999999999,2
MIDWEST,ACADEMIC,88605.36999999992,3
NORTHEAST,GROCERY,403710.6499999998,1


In [0]:
# Volume de vendas por marca e mês

sales_brand_month_df = (
    fact_sales.alias("f")
    .join(dim_brand.alias("b"), "brand_key")
    .join(dim_date.alias("d"), "date_key")
    .groupBy(
        "d.YEAR",
        "d.MONTH",
        "b.BRAND_NM"
    )
    .agg(
        sum("f.sales_volume").alias("total_sales_volume")
    )
    .orderBy("d.YEAR", "d.MONTH", col("total_sales_volume").desc())
)

display(sales_brand_month_df)

YEAR,MONTH,BRAND_NM,total_sales_volume
2006,1,LEMON,505845.2199999979
2006,1,RASPBERRY,392551.69999999914
2006,1,STRAWBERRY,315902.64999999927
2006,2,LEMON,550940.8499999979
2006,2,RASPBERRY,409661.5199999984
2006,2,STRAWBERRY,345518.39999999915
2006,3,LEMON,765942.5799999965
2006,3,RASPBERRY,587522.5599999976
2006,3,STRAWBERRY,475032.17999999865
2006,3,GRAPE,7.5


In [0]:
# Marca com menor volume de vendas por região

lowest_brand_region_df = (
    fact_sales.alias("f")
    .join(dim_region.alias("r"), "region_key")
    .join(dim_brand.alias("b"), "brand_key")
    .groupBy(
        "r.Btlr_Org_LVL_C_Desc",
        "b.BRAND_NM"
    )
    .agg(
        sum("f.sales_volume").alias("total_sales_volume")
    )
)

window_lowest = Window.partitionBy(
    "Btlr_Org_LVL_C_Desc"
).orderBy(
    col("total_sales_volume").asc()
)

lowest_brand_region_df = (
    lowest_brand_region_df
    .withColumn("ranking", row_number().over(window_lowest))
    .filter(col("ranking") == 1)
    .orderBy("Btlr_Org_LVL_C_Desc")
)

display(lowest_brand_region_df)

Btlr_Org_LVL_C_Desc,BRAND_NM,total_sales_volume,ranking
CANADA,STRAWBERRY,74009.61,1
GREAT LAKES,STRAWBERRY,208747.89999999985,1
MIDWEST,STRAWBERRY,149206.7799999999,1
NORTHEAST,STRAWBERRY,195581.48999999985,1
SOUTHEAST,RASPBERRY,223958.50000000003,1
SOUTHWEST,GRAPE,7.5,1
WEST,STRAWBERRY,134511.17999999988,1


In [0]:
# Gravação das tabelas analíticas de resposta do case

top_trade_groups_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.rpt_top3_trade_groups_by_region")

sales_brand_month_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.rpt_sales_by_brand_month")

lowest_brand_region_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.rpt_lowest_brand_by_region")

print("Tabelas de resposta do case criadas com sucesso.")

Tabelas de resposta do case criadas com sucesso.


In [0]:
# Market Share por marca

brand_share_df = (
    fact_sales.alias("f")
    .join(dim_brand.alias("b"), "brand_key")
    .groupBy(
        "b.BRAND_NM"
    )
    .agg(
        sum("sales_volume").alias("total_sales_volume")
    )
)

total_volume = brand_share_df.agg(
    sum("total_sales_volume")
).collect()[0][0]

brand_share_df = (
    brand_share_df
    .withColumn(
        "market_share_pct",
        round(
            col("total_sales_volume") / lit(total_volume) * 100,
            2
        )
    )
    .orderBy(
        col("total_sales_volume").desc()
    )
)

display(brand_share_df)

BRAND_NM,total_sales_volume,market_share_pct
LEMON,1822728.6500000057,41.91
RASPBERRY,1389735.779999997,31.96
STRAWBERRY,1136453.2299999949,26.13
GRAPE,7.5,0.0
